# Master Pendulum

## Pendulum Dynamics

- We consider a simple pendulum consisting of a mass $m$, with distance $L_{cm}$ from the pivot to the center of mass, and an inertia $I$ about the pivot point.

- The state of the pendulum can be described by its angle $\theta$ (measured from the vertical) and its angular velocity $\dot{\theta}$.

- The pednulum dynamics are affected by gravity $g$ and any external torque $\tau_{drive}$ applied at the pivot.

- The equations of motion for the pendulum can be derived using Newton's second law.

$$\begin{aligned}
\alpha &= \frac{d \omega}{dt} = \frac{d^2 \theta}{dt^2} \\
\alpha &= \frac{\tau}{I} \\
\alpha &= \frac{\tau_{grav} + \tau_{drive}}{I} \\
\alpha &= \frac{-m \cdot g \cdot L_{cm} \cdot \sin(\theta) + \tau_{drive}}{I} \\
\end{aligned}$$

Where:
- $\alpha$ is the angular acceleration,
- $\omega$ is the angular velocity,
- $\theta$ is the angle of the pendulum,
- $m$ is the mass of the pendulum,
- $g$ is the acceleration due to gravity,
- $l$ is the distance from the pivot to the center of mass of the pendulum,
- $\tau_{drive}$ is the external torque applied to the pendulum,
- $I$ is the moment of inertia of the pendulum about the pivot point

**Parameter Synchronization Strategy**

- The assumption for the inertia $I = m \cdot L_{eff}^2$ fails, since the mass distribution of the pendulum is not concentrated at a single point (the CoM) but is distributed over its geometry described by the FEM model.
$$ I_{FEM} \neq m_{FEM} \cdot L_{cm}^2 $$

- Therefore, the inertia $I$ computed by the FEM pendulum is directly synchronized to the equation-based and OpenSim pendulum implementations to ensure consistent dynamics across all pendulum models.

- For all three models to produce identical acceleartions under the same applied torques, the following parameters are synchronized from the FEM pendulum to the equation-based and OpenSim pendulums:

$$\alpha = \frac{\tau}{I} = \frac{\tau_{grav} + \tau_{drive}}{I} = \frac{-m \cdot g \cdot L_{eff} \cdot \sin(\theta) + \tau_{drive}}{I}$$

  - Same total mass $m$
  - Same effective length $L_{eff}$ (distance from pivot to CoM)
  - Same moment of Inertia $I$ (about the pivot point)

### 1. FEM Pendulum

- The FEM Pendulum acts as master pendulum since it provides the most detailed information about the mass distribution and inertia of the pendulum system.

#### 1.1 Mass Calculation
- The FEM pendulum computes the total mass by integrating the mass density over the pendulum's 2D geometry and multiplying by the thickness.

$$m = \int_{\Omega_p} \rho \cdot t \, dA = \rho \cdot t \cdot A$$

where
 - $\rho$ is the mass density,
 - $t$ is the thickness of the pendulum,
 - $A$ is the area of the pendulum's 2D geometry.

#### 1.2 Inertia and Center of Mass Calculation

1. Center of Mass Location

$$r_{CoM} = \frac{1}{m} \int_{\Omega_p} \rho \cdot t \cdot \vec r \, d\Omega_p$$

With the coordinates:

$$c_x = \frac{\int_{\Omega_p} \rho \cdot t \cdot x \, d\Omega_p}{m} ,\quad c_y = \frac{\int_{\Omega_p} \rho \cdot t \cdot y \, d\Omega_p}{m}$$

#### 1.3 Moment of Inertia around the pivot point

- Using the parallel axis theorem, the moment of inertia about the pivot point is computed as:

$$I = \int_{\Omega_p} \rho \cdot r^2 \, d\Omega_p$$

where
- $r = \sqrt{x^2 + y^2}$ is the distance from the pivot point
- $I$ includes both the rotational inertia about the center of mass and the contribution from the mass distribution relative to the pivot point

### 2. Modelica

Primary Incorrect Approach (assumption $I = m \cdot L_{cm}^2$):
```Modelica
equation
  der(q)    = omega;
  der(omega) = -(g / L_cm) * sin(q) + torque/(m * L_cm^2);  // WRONG!
```

Correct Modelica Implementation:
```Modelica
equation
  der(q)    = omega;
  der(omega) = -(m * g * L_cm / I) * sin(q) + torque / I;
```

- This uses now the full inertia $I$ and not only $mL_{eff}^2$

### 3. OpenSim

- OpenSim allows to define the mass $m$ and an inertia tensor $\mathbf{I}$ for a body.

- **Body:** Head with mass $m$ and inertia $I_{head} about its own center$

- **Joint:** Hinge joint at distance $L_{eff}$ from the center of mass of the head.

- **Effective Inertia Calculation:**
$$I_{head} + m \cdot L_{eff}^2 = I$$

- Joints are defined to specify the kinematic connections between different bodies of the model.

- The resulting model can be descriibed by a kinematic chain.

- In our case, a hinge joint is used to connect the pendulum head to a fixed base, allowing rotation about a single axis.

- Given FEM parameters $m, L_{eff}, I$ compute (reverse parallel axis theorem):

$$I_{head} = I - m \cdot L_{eff}^2$$


**OpenSim Model Setup:**
```python
        # Add pendulum head body and attach sphere geometry
        head_name = 'pendulum_head'
        head_mass = mp['mass']
        head_com = osim.Vec3(0, 0, 0)
        inertia = mp['inertia']
        head_inertia = osim.Inertia(0, 0, inertia) # About z-axis
        head = osim.Body(head_name, head_mass, head_com, head_inertia)
        head_geom = osim.Sphere(mp['r_head'])
        head_geom.setColor(osim.Vec3(0.2, 0.2, 0.8))
        head.attachGeometry(head_geom)
        model.addBody(head)

        # Create pin joint to connect head to base
        l = mp['length']
        base_translation = osim.Vec3(0, 0, 0)
        base_orientation = osim.Vec3(0, 0, 0)
        head_translation = osim.Vec3(0, l, 0)
        head_orientation = osim.Vec3(0, 0, 0)
        head_to_base = osim.PinJoint('head_to_base',
                                    base, base_translation, base_orientation,
                                    head, head_translation, head_orientation)
        model.addJoint(head_to_base)